In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
import numpy as np
from tensorflow.keras.models import load_model

def load_data_from_df(df, image_size=(66, 200)):
    images = []
    labels = []
    for _, row in df.iterrows():
        # Load and preprocess image
        img = cv2.imread(row['Path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (image_size[1], image_size[0]))
        img = img.astype(np.float32) / 255.0  # Normalize the image
        images.append(img)
        # Extract labels (steer angle and throttle)
        labels.append([row['SteerAngle'], row['Throttle']])
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.float32)

def evaluate_model(test, model_name, display_stats=False, data_owner ="chris"):
    # Load the model
    model = load_model(model_name, safe_mode=False)
    
    # Load the dataset CSV
    csv_path = os.path.join('data', test, 'robot_log.csv')
    df = pd.read_csv(csv_path, delimiter=';')
    
    # Define the image directory and update the DataFrame paths
    image_dir = os.path.join('data', test, 'IMG')

    if data_owner == "chris": 
        df['Path'] = df['Path'].str.replace(r'^C:\\Users\\Chris SB\\Downloads\\IMG\\', '', regex=True)
    elif data_owner == "eric": 
        df['Path'] = df['Path'].str.replace(r"^C:\\Users\\ericu\\Desktop\\Windows_Roversim\\Roversim_logging\\ZigZag_Driving\\IMG\\", '', regex=True)
    elif data_owner == "mark":
        df['Path'] = df['Path'].str.replace(r"^C:\\Mark's Python files\\ME5920\\HW3_Gardocki\\IMG\\", '', regex=True)

    
    df['Path'] = df['Path'].apply(lambda x: os.path.join(image_dir, x))
    
    # Shuffle the DataFrame
    df = df.sample(frac=1).reset_index(drop=True)
    
    # Optionally display one sample image (if available)
    if display_stats == True:
        img_path = df['Path'].iloc[20]
        print(f"Displaying image: {img_path}")
        print(f"Steering Angle: {df['SteerAngle'].iloc[20]}")
        print(f"Throttle: {df['Throttle'].iloc[20]}")
        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.show()
    
        # Plot histogram for Steering Angle distribution
        df['SteerAngle'].hist(bins=50)
        plt.title('Steering Angle Distribution')
        plt.show()
        
        # Plot histogram for Throttle distribution
        df['Throttle'].hist(bins=50)
        plt.title('Throttle Distribution')
        plt.show()
    
    # Select relevant columns
    df = df[['Path', 'SteerAngle', 'Throttle']]
    
    # Load test data (images and labels)
    X_test, y_test = load_data_from_df(df)
    
    # Evaluate the model on the test data
    loss = model.evaluate(X_test, y_test)
    print(f"Test Loss: {loss}")
    
    return loss



2025-04-08 01:24:12.699116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-08 01:24:12.714011: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-08 01:24:12.718980: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-08 01:24:12.731050: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
test_loss_1 = evaluate_model(test="test_1", model_name="test1model.keras")
test_loss_2 = evaluate_model(test="test_2", model_name="test1model.keras")
test_loss_3 = evaluate_model(test="test_3", model_name="test1model.keras")
test_loss_4 = evaluate_model(test="test_4", model_name="test1model.keras")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 + test_loss_4)/4
average_loss

62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 5.4712
Test Loss: 5.308021068572998
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 43.1256
Test Loss: 42.1873893737793
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 40.9459
Test Loss: 41.548160552978516
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 40.1561
Test Loss: 41.23072052001953


32.568572878837585

In [3]:
test_loss_1 = evaluate_model(test="test_1", model_name="test2model.keras")
test_loss_2 = evaluate_model(test="test_2", model_name="test2model.keras")
test_loss_3 = evaluate_model(test="test_3", model_name="test2model.keras")
test_loss_4 = evaluate_model(test="test_4", model_name="test2model.keras")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 + test_loss_4)/4
average_loss

62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 25.4435
Test Loss: 25.31366539001465
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 7.0805
Test Loss: 7.351560592651367


2025-04-07 02:11:37.431805: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 474091200 exceeds 10% of free system memory.


94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 31.8716
Test Loss: 31.91037940979004
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 40.9639
Test Loss: 40.84895706176758


26.356140613555908

In [4]:
test_loss_1 = evaluate_model(test="test_1", model_name="test3model.keras")
test_loss_2 = evaluate_model(test="test_2", model_name="test3model.keras")
test_loss_3 = evaluate_model(test="test_3", model_name="test3model.keras")
test_loss_4 = evaluate_model(test="test_4", model_name="test3model.keras")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 + test_loss_4)/4
average_loss

62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 20.6001
Test Loss: 20.402050018310547
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 30.3678
Test Loss: 29.189571380615234
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 4.8874
Test Loss: 4.941056251525879
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 31.6117
Test Loss: 32.354286193847656


21.72174096107483

In [5]:
test_loss_1 = evaluate_model(test="test_1", model_name="test4model.keras")
test_loss_2 = evaluate_model(test="test_2", model_name="test4model.keras")
test_loss_3 = evaluate_model(test="test_3", model_name="test4model.keras")
test_loss_4 = evaluate_model(test="test_4", model_name="test4model.keras")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 + test_loss_4)/4
average_loss

62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 20.2106
Test Loss: 19.707054138183594
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 33.8417
Test Loss: 34.03329086303711
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 22.5383
Test Loss: 22.156709671020508
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 4.9802
Test Loss: 5.116481781005859


20.253384113311768

In [7]:
test_loss_1 = evaluate_model(test="test_1", model_name="test5model.keras")
test_loss_2 = evaluate_model(test="test_2", model_name="test5model.keras")
test_loss_3 = evaluate_model(test="test_3", model_name="test5model.keras")
test_loss_4 = evaluate_model(test="test_4", model_name="test5model.keras")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 + test_loss_4)/4
average_loss

62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 26.3922
Test Loss: 25.77627944946289
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 48.5623
Test Loss: 48.281551361083984
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 32.4603
Test Loss: 32.68392562866211
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 40.9968
Test Loss: 39.518585205078125


36.56508541107178

In [3]:
test_loss_1 = evaluate_model(test="test_1", model_name="test6model.keras")
test_loss_2 = evaluate_model(test="test_2", model_name="test6model.keras")
test_loss_3 = evaluate_model(test="test_3", model_name="test6model.keras")
test_loss_4 = evaluate_model(test="test_4", model_name="test6model.keras")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 + test_loss_4)/4
average_loss

I0000 00:00:1743999044.360277  268720 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1743999044.440566  268720 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1743999044.440823  268720 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1743999044.442018  268720 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

58/62 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 20.2472

I0000 00:00:1743999051.293371  274822 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2025-04-07 04:10:53.143857: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_152', 176 bytes spill stores, 176 bytes spill loads



62/62 ━━━━━━━━━━━━━━━━━━━━ 7s 59ms/step - loss: 20.2059
Test Loss: 19.657106399536133


2025-04-07 04:10:58.902407: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 237283200 exceeds 10% of free system memory.
2025-04-07 04:10:59.173009: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 237283200 exceeds 10% of free system memory.


...

2025-04-07 04:11:06.523804: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 474091200 exceeds 10% of free system memory.


78/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 33.9961

2025-04-07 04:11:08.997267: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_152', 176 bytes spill stores, 176 bytes spill loads



94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - loss: 33.7339
Test Loss: 32.57758331298828
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 35.2441
Test Loss: 36.84348678588867


34.52785634994507

In [6]:
test_loss_1 = evaluate_model(test="test_4", model_name="test4model.keras")
test_loss_2 = evaluate_model(test="test_5", model_name="test4model.keras", data_owner= "eric")
test_loss_3 = evaluate_model(test="test_6", model_name="test4model.keras", data_owner= "mark")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 ) / 3
average_loss

87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 4.9876
Test Loss: 5.1164631843566895
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 38.1078
Test Loss: 38.486175537109375
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 23.0857
Test Loss: 22.457763671875


22.02013413111369

In [3]:
test_loss_1 = evaluate_model(test="test_4", model_name="test5model.keras")
test_loss_2 = evaluate_model(test="test_5", model_name="test5model.keras", data_owner= "eric")
test_loss_3 = evaluate_model(test="test_6", model_name="test5model.keras", data_owner= "mark")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 ) / 3
average_loss

87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 37.1290
Test Loss: 37.556392669677734
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5.5057
Test Loss: 5.476585865020752
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 19.5965
Test Loss: 19.64252471923828


20.891834417978924

In [4]:
test_loss_1 = evaluate_model(test="test_4", model_name="test6model.keras")
test_loss_2 = evaluate_model(test="test_5", model_name="test6model.keras", data_owner= "eric")
test_loss_3 = evaluate_model(test="test_6", model_name="test6model.keras", data_owner= "mark")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 ) / 3
average_loss

87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 37.0681
Test Loss: 36.843467712402344
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 32.3849
Test Loss: 31.641923904418945
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 2.1248
Test Loss: 2.07387638092041


23.519755999247234

In [4]:
test_loss_1 = evaluate_model(test="test_4", model_name="test10model.keras")
test_loss_2 = evaluate_model(test="test_5", model_name="test10model.keras", data_owner= "eric")
test_loss_3 = evaluate_model(test="test_6", model_name="test10model.keras", data_owner= "mark")
test_loss_4 = evaluate_model(test="test_7", model_name="test10model.keras", data_owner= "chris")
average_loss = (test_loss_1 + test_loss_2 + test_loss_3 + test_loss_4 ) / 4
average_loss

87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 6.1158
Test Loss: 6.179953098297119
113/113 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - loss: 6.7407
Test Loss: 6.7667460441589355
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 3.4581
Test Loss: 3.484724760055542
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - loss: 26.7837
Test Loss: 27.221023559570312


10.913111865520477